# Query Options

The ECB API shapes a request with **query parameters** after the key. This notebook covers the ones you reach for most: time range, latest observations, detail, and wildcards.

In [ ]:
using HTTP, CSV, Tables

base = "https://data-api.ecb.europa.eu/service/data"
ecb(flow, key; params = "format=csvdata") = CSV.File(HTTP.get("$base/$flow/$key?$params").body)

## Time Range

`startPeriod` and `endPeriod` follow the series frequency (`YYYY-MM-DD` for daily, `YYYY-MM` for monthly, `YYYY-Qn` for quarterly).

In [ ]:
usd2023 = ecb("EXR", "D.USD.EUR.SP00.A";
              params = "startPeriod=2023-01-01&endPeriod=2023-12-31&format=csvdata")
println("Rows in 2023: ", length(Tables.getcolumn(usd2023, :OBS_VALUE)))

## Latest Observation Only

`lastNObservations=1` returns just the most recent point — a current reading without the history.

In [ ]:
dfr = ecb("FM", "B.U2.EUR.4F.KR.DFR.LEV"; params = "lastNObservations=1&format=csvdata")
println("Current deposit facility rate: ",
        Tables.getcolumn(dfr, :OBS_VALUE)[1], "%")

## Discover Keys with `serieskeysonly`

Wildcard a dimension (leave it empty) and ask for keys only, to list every series that matches.

In [ ]:
# Every currency's daily reference rate — keys only, no observations
keys_only = ecb("EXR", "D..EUR.SP00.A"; params = "detail=serieskeysonly&format=csvdata")
unique(Tables.getcolumn(keys_only, :KEY))[1:min(end, 10)]

## Several Currencies in One Call

The `+` operator lists alternatives within a dimension.

In [ ]:
majors = ecb("EXR", "D.USD+GBP+JPY.EUR.SP00.A"; params = "lastNObservations=1&format=csvdata")
for (k, v) in zip(Tables.getcolumn(majors, :KEY), Tables.getcolumn(majors, :OBS_VALUE))
    println(k, " = ", v)
end

## Summary
- `startPeriod` / `endPeriod` bound the time range; `lastNObservations` / `firstNObservations` take the latest/oldest *n*.
- `detail=serieskeysonly` plus an empty (wildcard) dimension is the discovery trick.
- `+` returns several members of a dimension in a single request.